# Phase 4: Fine-Tuning Headline Sentiment Model

This notebook trains a sequence classification model (`distilbert/distilroberta-base`) on the **Financial PhraseBank** benchmark across **3 independent random seeds** (`42`, `43`, `44`).

### Methodological Guarantees:
1. **Strict Split Isolation**: Checkpoints are selected strictly on the validation split. Test sets are never touched during training.
2. **Multi-Seed Variance**: We report sample mean $\pm$ standard deviation ($\mu \pm \sigma$), not a single cherry-picked run.
3. **Domain Gap Measurement**: We evaluate on both the in-domain benchmark test split and general news headlines to quantify performance drop across domain boundaries.
4. **Free GPU Compatible**: Runs in under 15 minutes on a free Google Colab T4 or Kaggle P100 GPU.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Install Required Dependencies

In [ ]:
!pip install -q "transformers[torch]" datasets evaluate accelerate pyyaml scikit-learn

## 3. Clone Repository or Prepare Workspace

In [ ]:
import os
from pathlib import Path

# If running inside Colab and repo not yet cloned:
if not Path("configs/train.yaml").exists():
    !git clone https://github.com/Dhrupal-01/Sentiment-Analyzer.git
    %cd Sentiment-Analyzer

print("Current Directory:", os.getcwd())
assert Path("configs/train.yaml").exists(), "Configuration file configs/train.yaml not found!"

## 4. Inspect Training Configuration

In [ ]:
import yaml

with open("configs/train.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, sort_keys=False))

## 5. Execute Multi-Seed Fine-Tuning Pipeline

This runs `scripts/train.py` across all seeds (`42`, `43`, `44`). Each run selects the best checkpoint by `macro_f1` on the validation split.

In [ ]:
# Run training pipeline with GPU
!python scripts/train.py --config configs/train.yaml

## 6. Inspect Training Metrics and Run Summary

In [ ]:
import json
import pandas as pd

# Read step telemetry
metrics_df = pd.read_csv("reports/training/metrics.csv")
print("Training Telemetry (First 10 steps):")
display(metrics_df.head(10))

# Read run summary
with open("reports/training/run_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

print("\n--- Multi-Seed Results Summary ---")
print(f"Best Seed Checkpoint: Seed {summary['best_seed']}")
val_m = summary['validation_metrics']
test_m = summary['financial_phrasebank_test_metrics']
id_m = summary['in_domain_news_test_metrics']
gap = summary['domain_gap']

print(f"Validation Macro-F1:     {val_m['macro_f1_mean']:.3f} ± {val_m['macro_f1_std']:.3f}")
print(f"FPB Test Macro-F1:       {test_m['macro_f1_mean']:.3f} ± {test_m['macro_f1_std']:.3f}")
print(f"In-Domain News Macro-F1: {id_m['macro_f1_mean']:.3f} ± {id_m['macro_f1_std']:.3f}")
print(f"Domain Gap Drop:         {gap['macro_f1_drop']:+.3f} Macro-F1 ({gap['relative_f1_degradation_pct']:.1f}% relative degradation)")

## 7. Zip and Download Best Model Checkpoint

In [ ]:
!zip -r fine_tuned_model.zip models/fine_tuned/best reports/training

# If running inside Google Colab:
try:
    from google.colab import files
    files.download("fine_tuned_model.zip")
    print("Downloaded fine_tuned_model.zip successfully!")
except ImportError:
    print("Archive created at fine_tuned_model.zip for download.")

## 8. Optional: Push to Hugging Face Hub

In [ ]:
# To push to HF Hub, set your token:
# import os
# os.environ["HF_TOKEN"] = "your_write_token_here"
# !python scripts/train.py --config configs/train.yaml --push-to-hub